<a href="https://colab.research.google.com/github/kwb95124/steam-front-curator/blob/main/SteamFrontCurator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install 'tabulate' (a PyPI package for clean console tables - satisfies the outside tool requirement offline)
!pip install -q tabulate

import pandas as pd
from tabulate import tabulate

# Load the dataset from Colab session storage
df = pd.read_csv("steam_games.csv")

# Verify the dataset loaded properly
print(f"Success! Loaded {len(df)} rows and {len(df.columns)} columns.")
display(df.head(3))

Success! Loaded 1000 rows and 26 columns.


,appid,name,release_date,release_year,price,discount,required_age,windows,mac,linux,...,positive,negative,pct_pos_total,num_reviews_total,estimated_owners,owners_low,avg_playtime_hours,peak_ccu,short_description,critic_reviews
0,730,Counter-Strike 2,2012-08-21,2012,0.0,0,0,True,False,True,...,7480813,1135108,86,8632939,100000000 - 200000000,100000000,553.2,1212356,"For over two decades, Counter-Strike has offer...",NaN
1,578080,PUBG: BATTLEGROUNDS,2017-12-21,2017,0.0,0,0,True,False,False,...,1487960,1024436,59,2513842,50000000 - 100000000,50000000,0.0,616738,Play PUBG: BATTLEGROUNDS for free. Land on str...,NaN
2,570,Dota 2,2013-07-09,2013,0.0,0,0,True,True,True,...,1998462,451338,81,2452595,200000000 - 500000000,200000000,717.2,555977,"Every day, millions of players worldwide enter...",“A modern multiplayer masterpiece.” 9.5/10 – D...


In [2]:
class SteamCurator:
  """Analyzes Steam game data across ratings, accessibility, and player engagement."""

  def __init__(self, dataframe):
    self.df = dataframe.copy()

  def get_top_rated_games(self, min_reviews=100000, top_n=10):
    """Sub-Question 1: Filters games with >= 80% positive reviews and high review counts."""
    filtered = self.df[
        (self.df["num_reviews_total"] >= min_reviews)
        & (self.df["pct_pos_total"] >= 80)
    ]
    return filtered.sort_values(
        by=["pct_pos_total", "num_reviews_total"], ascending=[False, False]
    ).head(top_n)

  def evaluate_accessibility(self, game_subset, min_platforms=2):
    """Sub-Question 2 (Original Idea): Counts supported OS platforms (Windows/Mac/Linux)

    and filters for multi-platform games with strong language support.
    """
    subset = game_subset.copy()
    # Count how many operating systems (True = 1, False = 0) each game supports
    subset["os_count"] = (
        subset["windows"].astype(int)
        + subset["mac"].astype(int)
        + subset["linux"].astype(int)
    )
    accessible = subset[subset["os_count"] >= min_platforms]
    return accessible.sort_values(
        by=["os_count", "n_languages"], ascending=[False, False]
    )

  def rank_by_player_engagement(self, game_subset):
    """Sub-Question 3: Ranks candidate games by Peak CCU and average playtime hours."""
    return game_subset.sort_values(
        by=["peak_ccu", "avg_playtime_hours"], ascending=[False, False]
    )


# Verify the class initializes properly
test_curator = SteamCurator(df)
print("SteamCurator class defined and verified!")

SteamCurator class defined and verified!


In [3]:
def format_table(dataframe, columns, headers):
  """Uses the external 'tabulate' PyPI package to render clean, readable tables."""
  return tabulate(
      dataframe[columns], headers=headers, tablefmt="github", showindex=False
  )


print("Formatting helper function ready!")

Formatting helper function ready!


In [4]:
def run_steam_curator_app(dataframe):
  print("=================================================================")
  print("           STEAM STORE FRONT PAGE CURATOR APPLICATION            ")
  print("=================================================================")
  curator = SteamCurator(dataframe)

  # --- SUB-QUESTION 1 ---
  print(
      "\nSUB-QUESTION 1 (Ratings): Which games have the highest player ratings"
      " among titles with >= 100,000 reviews?"
  )
  top_rated = curator.get_top_rated_games(min_reviews=100000, top_n=8)
  print(
      format_table(
          top_rated,
          columns=["name", "pct_pos_total", "num_reviews_total", "genres"],
          headers=["Game Title", "Positive %", "Total Reviews", "Genres"],
      )
  )

  # --- SUB-QUESTION 2 (OWN IDEA) ---
  print(
      "\nSUB-QUESTION 2 (Accessibility - Original Idea): Which of these"
      " top-rated games support at least 2 operating systems and the most"
      " languages?"
  )
  accessible_games = curator.evaluate_accessibility(top_rated, min_platforms=2)
  print(
      format_table(
          accessible_games,
          columns=[
              "name",
              "windows",
              "mac",
              "linux",
              "os_count",
              "n_languages",
          ],
          headers=[
              "Game Title",
              "Win",
              "Mac",
              "Linux",
              "Total OS",
              "Languages",
          ],
      )
  )

  # --- SUB-QUESTION 3 ---
  print(
      "\nSUB-QUESTION 3 (Player Engagement): Which of these accessible,"
      " top-rated games have the highest Peak CCU and average playtime?"
  )
  final_ranking = curator.rank_by_player_engagement(accessible_games)
  print(
      format_table(
          final_ranking,
          columns=[
              "name",
              "peak_ccu",
              "avg_playtime_hours",
              "pct_pos_total",
              "n_languages",
          ],
          headers=[
              "Game Title",
              "Peak CCU",
              "Avg Playtime (Hrs)",
              "Positive %",
              "Languages",
          ],
      )
  )

  # --- FINAL RECOMMENDATION ---
  print("\n=================================================================")
  print("        FINAL RECOMMENDATION FOR STEAM STORE FRONT PAGE          ")
  print("=================================================================")
  top_3 = final_ranking.head(3)
  for rank, (_, row) in enumerate(top_3.iterrows(), 1):
    print(
        f"{rank}. {row['name']} — {row['pct_pos_total']}% positive reviews, "
        f"supports {row['os_count']}/3 OS platforms & {row['n_languages']} languages, "
        f"with {row['peak_ccu']:,} Peak CCU and {row['avg_playtime_hours']} hrs avg playtime."
    )


# Run the complete application
run_steam_curator_app(df)

           STEAM STORE FRONT PAGE CURATOR APPLICATION            

SUB-QUESTION 1 (Ratings): Which games have the highest player ratings among titles with >= 100,000 reviews?
| Game Title             |   Positive % |   Total Reviews | Genres                                 |
|------------------------|--------------|-----------------|----------------------------------------|
| Stardew Valley         |           98 |          733626 | Indie, RPG, Simulation                 |
| People Playground      |           98 |          258453 | Action, Casual, Indie, Simulation      |
| Hades                  |           98 |          255511 | Action, Indie, RPG                     |
| Vampire Survivors      |           98 |          234756 | Action, Casual, Indie, RPG             |
| Balatro                |           98 |          108031 | Casual, Indie, Strategy                |
| Terraria               |           97 |         1102434 | Action, Adventure, Indie, RPG          |
| Euro Truck Simu